<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/4-2_tracing-medical-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>4.2-Tracing and Evaluating LLMs with LangSmith.
  </h2>
    <h3>Tracing a Medical Agent with LangSmith</h3>
    <p>by <b>Pere Martra</b></p>
</div>


In this notebook, we will explore how to trace the different calls that occur in a LangChain Agent using LangSmith. We will use a familiar agent, employed in the LangChain section of the course, where a RAG system with medical information was created.

So, not only will we observe the traces of the agent, but we will also examine the traces of the retriever. Additionally, we'll inspect the query sent to the vectorial database and the returned results.
__________

#Installing libraries & Loading Dataset

In [ ]:
# === Colab dependency guard (bban4040) ===
# The langchain / langsmith stack upgrades transitive packages (requests,
# opentelemetry-*) past the exact versions Colab's preinstalled google
# packages pin (google-colab, google-adk, the otlp/gcp exporters), which
# prints noisy "pip's dependency resolver ... is incompatible" errors.
# We pin those families to the versions already installed so the installs
# below leave them untouched. PIP_CONSTRAINT is honored by every %pip call
# in this kernel. Harmless off Colab (nothing matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")


In [ ]:
# This notebook runs on the langchain 1.x line (langchain + langchain-classic).
# langchain-community is sunset/deprecated, so we don't use it: Chroma now lives
# in langchain-chroma, and the DataFrameLoader is replaced with direct Document
# construction.
%pip install -q langchain
%pip install -q langchain-classic
%pip install -q langchain-openai
%pip install -q langchain-chroma
%pip install -q datasets

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


# LangSmith tracing keys (resolve LANGCHAIN_API_KEY or LANGSMITH_API_KEY;
# mirrors the OPENAI_API_KEY pattern so this is Colab-ready once the secret
# is added). Tracing only activates if a key is present.
_ls_key = get_secret("LANGCHAIN_API_KEY") or get_secret("LANGSMITH_API_KEY")
if _ls_key:
    os.environ["LANGCHAIN_API_KEY"] = _ls_key
    os.environ["LANGSMITH_API_KEY"] = _ls_key


We will download the dataset from the Hugging Face datasets library. It's a dataset with information about diseases.

In [ ]:
from datasets import load_dataset

data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split='train')


In [ ]:
data = data.to_pandas()
data.head(10)

In [ ]:
#uncoment this line if you want to limit the size of the data.
data = data[0:100]

As you can see, the medical information in the dataset is well-organized, and to someone like me, who is not an expert in the field, it appears to be quite valuable. This information could be a useful addition to any general medicine book to support primary care doctors.

Load the langchain libraries to load the document.

In [ ]:
# Chroma moved out of langchain-community into the dedicated langchain-chroma
# package; Document comes from langchain-core (replaces the community DataFrameLoader).
from langchain_chroma import Chroma
from langchain_core.documents import Document

The Document is in the Answer column, and the others columns are Metadata.

In [ ]:
# Equivalent of DataFrameLoader(data, page_content_column="Answer"): one Document
# per row, page content from the "Answer" column and every other column as metadata.
df_document = [
    Document(
        page_content=row["Answer"],
        metadata={col: row[col] for col in data.columns if col != "Answer"},
    )
    for _, row in data.iterrows()
]

In [ ]:
display(df_document[:2])

We can chunk the documents. The size to which we want to split the document is a design decision. The larger it is, the larger the prompt will be, and the slower the Model's response process.

We also need to consider the maximum prompt size and ensure that the document does not exceed it.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1250,
                                      separator="\n",
                                      chunk_overlap=100)
texts = text_splitter.split_documents(df_document)


These warnings we see are because it can't perform the partition of the required size. This is because it waits for a page break to divide the text and does so when possible.

In [ ]:
first_doc = texts[1]
print(first_doc.page_content)

### Initialize the Embedding Model and Vector DB

We load the text-embedding-ada-002 model from OpenAI.

In [ ]:
import os

# OPENAI_API_KEY is resolved from Colab Secrets / env / .env in the portable-setup
# cell above. Re-resolve defensively here and only assign it when we actually get
# a non-empty string -- os.environ values must be strings, and get_secret may
# return None (or a non-string) when nothing is configured. We never prompt
# interactively, so the notebook stays runnable unattended (HPC/CI/headless).
if not os.environ.get("OPENAI_API_KEY"):
    _key = get_secret("OPENAI_API_KEY")
    if isinstance(_key, str) and _key.strip():
        os.environ["OPENAI_API_KEY"] = _key.strip()
    else:
        print("No OPENAI_API_KEY found -- set it as a Colab Secret, env var, "
              "or .env entry before running the cells that call OpenAI.")

Obtain Your LangChain API Key from your Personal->Settings Area in LangSmith panel.

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/langsmith_API_KEY.jpg?raw=true)

In [ ]:
# LangSmith tracing is optional. Resolve a key only from secrets/env -- never
# prompt for it, so the notebook runs unattended (e.g. on an HPC cluster or in
# a headless CI run). If no key is present, tracing stays disabled below.
_ls_key = os.environ.get("LANGCHAIN_API_KEY") or os.environ.get("LANGSMITH_API_KEY")
if _ls_key:
    os.environ["LANGCHAIN_API_KEY"] = _ls_key

In [ ]:
# Enable LangSmith tracing only when an API key is available. Without a key the
# LangSmith endpoint would reject every call, so we explicitly turn tracing off
# and the rest of the notebook runs exactly the same -- just untraced.
if _ls_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = "langsmith_test3"
    print("LangSmith tracing ENABLED (project: langsmith_test3).")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ["LANGSMITH_TRACING"] = "false"
    print("No LANGCHAIN_API_KEY found -- LangSmith tracing DISABLED. "
          "The agent still runs; calls just won't be sent to LangSmith.")

In [ ]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name
)

The execution of this cell may take 3 to 5 minutes. If you want it to be faster, you can reduce the number of records in the dataset.

In [ ]:
# Colab Drive path is not portable; use a local dir that works everywhere.
directory_cdb = os.environ.get('CHROMA_DIR', './chromadb')
chroma_db = Chroma.from_documents(
    df_document, embed, persist_directory=directory_cdb
)

We are going to create three objects.

* The language model, which can be any of those from OpenAI, the most common being gpt-3.5.
* The memory, responsible for keeping the prompt with all the necessary history.
* The retrieval, used to obtain information stored in ChromaDB.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import RetrievalQA

llm=OpenAI(temperature=0.0)

conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=4, #Number of messages stored in memory
    return_messages=True #Must return the messages in the response.
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=chroma_db.as_retriever()
)

We can try the isolated Retrieval to see if the information it returns is relevant.




In [ ]:
qa.invoke("What is the main symptom of LCM?")

When observing the Retriever on Langsmith is possible to see the input and the documents returned by it:
![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_Retriever_1.jpg?raw=true)

In the image below you can observe the call to the Model where the full prompt and the response are displayed.
![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_Retriever_2.jpg?raw=true)


## Creating the Agent.

In [ ]:
from langchain_classic.agents import Tool

# RetrievalQA.run is deprecated; wrap .invoke and return the answer string so the
# tool still hands the agent a plain string (RetrievalQA's output key is "result").
def medical_kb(query: str) -> str:
    return qa.invoke(query)["result"]

#Defining the list of tool objects to be used by LangChain.
tools = [
    Tool(
        name='Medical KB',
        func=medical_kb,
        description=(
            """use this tool when answering medical knowledge queries to get
            more information about the topic"""
        )
    )
]

In [ ]:
from langchain_classic.agents import create_react_agent
from langchain_core.prompts import PromptTemplate

# Inline copy of the public "hwchase17/react-chat" prompt. In LangChain 1.x
# hub.pull() of public prompts is disabled (untrusted serialized objects), so
# we define the same ReAct-chat template directly to keep the notebook runnable.
react_chat_template = """Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussion on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful system that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:

```
Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
```

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

```
Thought: Do I need to use a tool? No
Final Answer: [your response here]
```

Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_chat_template)
agent = create_react_agent(
    tools=tools,
    llm=llm,
    prompt=prompt,
)

In [ ]:
# Create an agent executor by passing in the agent and tools
from langchain_classic.agents import AgentExecutor
agent_executor2 = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True,
                               memory=conversational_memory,
                               max_iterations=30,
                               max_execution_time=600,
                               handle_parsing_errors=True
                               )

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

First i will try a request not related to the Medical field.

In [ ]:
agent_executor2.invoke({"input": "What is 2 multiplied by 2?"})

The initial call to the Agent happens with just one call to OpenAI. One piece of information available in LangSmith is the entire prompt. I'll copy it just below the image, so you can see.

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_1AE_1.jpg?raw=true)



```
Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful tool that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

Medical KB: use this tool when answering medical knowledge queries to get
            more information about the topic

To use a tool, please use the following format:


Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [Medical KB]
Action Input: the input to the action
Observation: the result of the action


When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:


Thought: Do I need to use a tool? No
Final Answer: [your response here]


Begin!

Previous conversation history:
[]

New input: What is 2 multiplied by 2?
```

Perfect, the model has responded without accessing the configured knowledge database.

Now I will try with a question that is also not related to health.

In [ ]:
agent_executor2.memory.clear()

In [ ]:
agent_executor2.invoke({"input": """I have a patient that can have Botulism,
how can I confirm the diagnosis?"""})

Perfect, the most important thing for us is that it has been able to identify that it should go to the medical database to search for information about the symptoms.
![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_1AE_3.jpg?raw=true)

In [ ]:
agent_executor2.invoke({"input": "Is this an important illness?"})

On the left side of the image, you can see the various calls made by the Agent, which took 2.5 seconds to execute and consumed 2000 tokens.

I will try to describe what happens in each call.

**OpenAI**: he complete prompt, including the template from Hugging Face, and the user's question is passed to the OpenAI Model. It responds with the following step. Its answer is:

*Thought: Do I need to use a tool?*

*Yes  Action: Medical KB*  

*Action Input: Botulism*

**Medical KB**: The Agent utilizes the configured tool, passing only a single word as a parameter: "Botulism."

**Medical KB.Retriever:** The retriever returns four documents extracted from the Vectorial Database.

**Medical KB.OpenAI:** The prompt is constructed with the information from the vectorial database. Even though I won't include the paste, I manage to identify that the four returned documents are actually two but duplicated. This, in a real project, could have helped me detect some issue, perhaps I have duplicates in the dataset, or maybe I loaded them twice. In any case, I'm consuming many more tokens than necessary. The model returns a response created considering the information contained in the prompt.

**OpenAI:** In this final call to the model, it decides that the received response is what the user needs. Therefore, it marks it as correct and returns it to the user.

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_1AE_3.jpg?raw=true)

And the memory works perfectly. We can maintain a conversation, taking into account that the model knows the previous questions and answers.


# Conclusions.
LangSmith is an incredibly useful tool for tracing and storing all the information generated when making calls to LangChain.

The experiment has been a small success. The Vectorial database has been configured and filled with information from the dataset. A LangChain agent has been created, and it has been able to retrieve information from the database only when necessary. Don't forget that our ChatBot has memory.



And you have all the information in a project stored in LangSmith!!!!
![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_1AE_Final.jpg?raw=true)



---